
AI-Powered HR Assistant: A Use Case for Nestle’s HR Policy Documents

Overview:<br/>
The project aims to create a conversational chatbot that responds to user inquiries using PDF document information. It requires proficiency in extracting and converting text into numerical vectors, establishing an answer-finding mechanism, and designing a user-friendly chatbot interface with Gradio. Additionally, the initiative emphasizes structuring inquiries for clear communication and deploying the chatbot for practical use, guaranteeing the system's accessibility and efficiency in meeting user needs.
Instructions
•	Review the learning materials and the Gradio documentation provided for the project.
•	Read the sections on situation, task, action, and result carefully to understand the assignment.
•	Complete and submit the assignment through the Learning Management System (LMS).
•	Adhere closely to the provided guidelines, ensuring your submission contains all necessary analyses and interpretations.

Situation:<br/>
As a developer, you have received the critical task of improving the operational efficiency of Nestlé's human resources department, a leading multinational corporation. Your toolkit includes cutting-edge conversational AI technology, Python libraries, the powerful GPT model from OpenAI, and the user-friendly Gradio UI. Your mission is to integrate these advanced tools seamlessly to transform HR processes, creating a more streamlined and efficient workflow within the Nestlé organization.

Task: <br/>
Your task is to develop a conversational chatbot. This chatbot must answer queries about Nestlé's HR reports efficiently. Use Python libraries, OpenAI's GPT model, and Gradio UI. These tools will help you create a user-friendly interface. This interface will extract and process information from documents. It will provide accurate responses to user queries.

Action: <br/>
•	Import essential tools and set up OpenAI's API environment.
•	Load Nestle's HR policy using PyPDFLoader and split it for easy processing.
•	Create vector representations for text chunks using Chroma dB and OpenAI's embeddings.
•	Build a question-answering system using the GPT-3.5 Turbo model to retrieve answers from text chunks.
•	Create a prompt template to guide the chatbot in understanding and responding to users.
•	Use Gradio to build a user-friendly chatbot interface, enabling interaction and information retrieval.

Result: <br/>
Upon completing this project, you will submit an IPYNB file demonstrating your ability to use advanced AI and machine learning technologies to develop a conversational chatbot. Your submission must include the entire workflow: setting up the programming environment, processing text documents, creating text vector representations, and building a question-answering system. Ensure the interface is user-friendly to facilitate effective interaction and information retrieval. 

#Solution Brief:
1. I setup the program as shown below.
2. Used LLamaIndex Parser to parse PDF document.
3. Used "BAAI/bge-large-en-v1.5" model to create Embedding. 
4. Used FAISS VectorStore to Store Embeddings.
5. Once Chunks were retrieved we used Cross-Encoding to score and retrieve top three document.
6. Those documetns we passed to Chat LLM (llama3.1:8b-instruct-q4_K_M) to create concise output.

Test Cases:<br/>

Hi, I am HR Policy Assistant. Please ask your question.<br/>

User:  Get me Information on Leave Policy<br/>
Searching Please wait...<br/>

Assistant -  I could not find that information. The retrieved contexts do not mention anything about leave policies, but they do discuss employment conditions, total rewards, employee relations, and organizational structure. If you would like to ask a different question or provide more context, I'll be happy to help.<br/>

User:  What about Travel Policy ?<br/>
Searching Please wait...<br/>

Assistant -  I could not find that information in the Retrieved Contexts.<br/>

User:  Ok, what about Training Policy ?<br/>
Searching Please wait...<br/>

Assistant -  Answer: <br/>
The Company determines training and development priorities. The responsibility for turning these into actions is shared between employees, line managers, and the Human Resources.<br/>

Experience and on-the-job training are the primary source of learning. Managers are responsible for guiding and coaching employees to succeed in their current positions.<br/>

Nestlé also offers a comprehensive range of training activities and methodologies to support everyone’s learning and growth. Attending a programme should never be considered as a reward but as a component of on-going development.<br/>

Additionally, corporate leadership programmes help us develop and retain the best-qualified management. Leaders have the opportunity to attend either international training courses at Rive-Reine, which build integrated business understanding and solidify and reinforce Nestlé values and principles, or programmes conducted by our strategic learning partners.<br/>




In [10]:
# Import System Packages
import os
import keyboard
from dotenv import load_dotenv
from pathlib import Path


# Import Langchain Agents Modules.
from langchain_core.messages import SystemMessage, HumanMessage


# Import Langchain Ollama Chatbot Interfaces
from langchain_ollama import ChatOllama

# Import Sentence Embedding Transformer
import textwrap as tw
from sentence_transformers import SentenceTransformer, CrossEncoder

# Import RAG DB FAISS
import faiss
# from llama_parse import LlamaParse
from llama_cloud import LlamaCloud
from llama_index.core import Document, StorageContext, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter   # MarkdownNodeParser is efficient if expand[..] is markdown  
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.faiss import FaissVectorStore


# Huggingface Cache Folder
hf_cache_folder = os.path.expanduser("~/.cache/huggingface/hub")

# Load Env Info
load_dotenv()


True

In [11]:
# Parse Documents (PDF, Doc etc...)
# Define Document Parser and Chunkng Strategy

llamaClient = LlamaCloud(
    api_key = os.getenv("LLAMA_CLOUD_API_KEY")
)

# Define Chunking Strategy
chunk_strategy = SentenceSplitter(
    chunk_size=500,
    chunk_overlap=50
)


In [12]:
# Create Embedding for the Document Nodes
# Instantiate FAISS Vector Store
# Instantiate StorageContext associated with LlamaParse
# Store Doc node in FAISS using VectorStoreIndex of LlamaParse

embedding_llm = HuggingFaceEmbedding(
    model_name = "BAAI/bge-large-en-v1.5",
    cache_folder = hf_cache_folder
)

# Instantiate FAISS with Embedding Dimensions
dimension = 1024       #  Dimension BAAI Large(1024) and BAAI Small(384)
faiss_index = faiss.IndexFlatL2(dimension)
faiss_vector_store = FaissVectorStore(faiss_index = faiss_index)

# Instantiate LlamaParse Storage Context
storage_context = StorageContext.from_defaults(
    vector_store = faiss_vector_store
)

llama_vector_store_index = VectorStoreIndex(
    nodes = [],
    embed_model = embedding_llm,
    storage_context = storage_context
)

# Create a Retriever Engine for Simantic Search to get the Doc Chunk, Scores & Metadata
# as_query_engine function does not provide the Simlarity Scores, but provide final Answer.
# query_engine = llama_vector_store_index.as_query_engine(similarity_top_k=5)
query_engine = llama_vector_store_index.as_retriever(similarity_top_k=5)


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 8181.13it/s]


In [13]:
# Create RAG Bot for Generate LLM Response.

# Initialized the RAG LLM
def create_rag_llm(model_name: str) -> ChatOllama:
    
    ol_rag_llm = ChatOllama(
        model = model_name,
        temperature = 0.0,
        num_predict = 8192,
        num_ctx = 8192,
        model_kwargs = {
            "repeat_penalty": 1.1,
            "options": {
                "use_cache": False,
                "use_mmap": False
            }
        }
    )

    return(ol_rag_llm)

In [14]:
# Insert to Vector Store Funcation
# Retrieve from Vector Store Funcation

# Insert Individual Doc Sections / Pages that was Parsed.
def insert_parsed_doc(file_name: str, page_num: int, doc_text: str):
    try:
        
        # Wrap Doc / Pages into LlamaIndex Doc.
        llama_doc = Document(
            text = doc_text,
            metadata = {
                "file_name": file_name,
                "page_number": page_num
            }
        )

        # Instert Docuemtns to FAISS
        doc_chunks = chunk_strategy.get_nodes_from_documents([llama_doc])   # GetNode Function need list as argument.
        llama_vector_store_index.insert_nodes(doc_chunks)
        
        total_embedding = llama_vector_store_index.storage_context.vector_store._faiss_index.ntotal
        print("Total Embddings: ", total_embedding, flush=True)

        return(True)
    
    except Exception as exp:
        print("Error Insert Doc: ", type(exp).__name__)
        return(False)


# Retrieve Relevent Chunks from the Vector Store.
def retrieve_doc_chunks(query):
    try:
        resp_chunks = query_engine.retrieve(query)

        #for chunk in resp_chunks:
        #    print("Retrieve Score: ", chunk.score)
        #    print("Retrieve Metadata: ", chunk.metadata.get("file_name"), ":", chunk.metadata.get("page_number"))
        #    print("Retrieve Text: ", chunk.text)
        #    print(resp_chunks[0].text)
        
        return(resp_chunks)
    
    except Exception as exp:
        return(f"Error Retrieve Doc: {exp}")


# Instantiate HF Cross-Encoder Model     
# Setup Re-Ranking after Chunk Retrieval. 
# Respond with a List Chunks

cross_encoder = CrossEncoder(
    model_name_or_path = "cross-encoder/ms-marco-MiniLM-L-6-v2",
    cache_folder = hf_cache_folder
)

def rerank_doc_chunks(query: str, doc_chunks: list) -> list:
    try:

        # Create a Query & Chunk Pair for every Chunk
        doc_pair = []
        for chunk in doc_chunks:
            doc_pair.append([query, chunk.text])

        rerank_scores = cross_encoder.predict(
            inputs = doc_pair,
            device = "cpu",
            convert_to_numpy = True
        )

        # print("Rerank Score: ", rerank_scores, flush=True)
        
        return(rerank_scores)
    
    except Exception as exp:
        return(f"Error Re-Ranking: {exp}")


The Transformer `cache_dir` argument is deprecated. Please pass `cache_dir` via `model_kwargs`, `processor_kwargs`, and/or `config_kwargs` instead.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 20248.36it/s]


In [15]:
# Compose & Generate Response from selected Chunks 
# Initialise Qwen 7B Quantized LLM using Ollama
# Use the Local LLM to Generate the Response

def format_doc_chunks(query: str, chunks: list) -> str:
    try:
        ctx_list = []
        ctx_template = "Document Chunk-{}: {}"
        
        for i, chunk in enumerate(chunks):
            ctx_list.append(ctx_template.format(i+1, chunk))

        # Formatting Query & Contexts
        context_str = "Retrieved Contexts: \n"
        context_str = context_str + "\n".join(ctx_list) + "\n\n"
        context_str = context_str + "Question: " + query        # Question Alway give at end.

        # print("RAG Context: \n", context_str)

        return(context_str)
    
    except Exception as exp:
        return(f"Error formatting RAG Context: {exp}")


# Generate RAG Response.
def generate_rag_response(query: str, doc_chunks: list, ragbot: ChatOllama) -> str:

    chat_chain = []

    system_prompt = """
    1. You are a Research Assistant.
    2. Use ONLY the information provided in the Retrieved Context to provide a Response.
    3. If the answer cannot be found in the Retrieved Contexts, say: 'I could not find that information'
    
    Critical Rules for RAG Response:
    1. Do not make up facts.
    2. Do not use Outside Knowledge.
    3. If multiple retrieved documents contain relevant information, combine them into a single coherent answer.
    4. Provide response to Question in 'Answer' variable.
    """

    rag_context = format_doc_chunks(query, doc_chunks)

    chat_chain = [SystemMessage(content=system_prompt)]
    chat_chain.append(HumanMessage(content=rag_context))

    ai_message = (ragbot.invoke(chat_chain).content).split("<|im_start|>assistant")[-1]

    # print("RAG AI Message: ", ai_message)
    
    return(ai_message)


In [16]:

# Research Papers Search Tool
def search_hrpolicy_data(user_query: str, ragbot: ChatOllama) -> str:
    """ Search the HR Policy Database to Answer the User Query.
    Args:
        user_query: The exact question or request from user in plain text.
    Return: 
        response: A short factual answer for the question from the vector store.
    """
    print("Searching Please wait...")

    # Retrieve Relevent Chunks from Vector Store
    retrieved_chunks = retrieve_doc_chunks(user_query)

    # Format Chunks with Re-Rank Relevance Score & Sort Descend
    rerank_scores = rerank_doc_chunks(user_query, retrieved_chunks)
    chunk_relevences = [(retrieved_chunk, float(rerank_score)) 
                        for retrieved_chunk, rerank_score in zip(retrieved_chunks, rerank_scores)
    ]
    chunk_relevences.sort(key=lambda item: item[1], reverse=True)

    # print("Relevent Chunks: ", list(chunk_relevences))

    # Choose Chunks with Top 3 Scores
    j = 0
    top3_chunks = []
    for chunk, score in chunk_relevences:
        if ((j < 5)):
            top3_chunks.append(chunk.text)
            j = j + 1

    # Invoke Generate RAG Response
    rag_response = generate_rag_response(user_query, top3_chunks, ragbot)

    return(rag_response)



In [17]:
# Load Documents into Vector Store using LlamaCloud Parser
# Each file is Read, uploaed to LLamaParse Cloud.
# Files are not Cached, generate Markdowns from PDF.
# Each File is processed one at a time in a loop.

async def parse_docs(client: LlamaCloud):
    folder_path = Path("C:\\RanjithC\\AIProjects\\PromptEngg\\huggingface\\data\\simpli_learn\\hr_policy\\")
    file_list = [str(file) for file in folder_path.glob("*.pdf")]

    # Read Each File in a Loop & Parse to multiple Docs
    for file_path in file_list:
        print("File Parsing: ", file_path)

        upload_file = client.files.create(file=file_path, purpose="parse")
        parse_result = client.parsing.parse(
            file_id =  upload_file.id, 
            tier = "cost_effective", 
            version = "latest",
            disable_cache = True,
            expand = ["markdown", "metadata"]
        )

        for doc in parse_result.markdown.pages:
            print("Upload: ", doc)
            insert_parsed_doc(Path(file_path).name, doc.page_number, doc.markdown)

    return(True)

await parse_docs(llamaClient)

File Parsing:  C:\RanjithC\AIProjects\PromptEngg\huggingface\data\simpli_learn\hr_policy\1728286846_the_nestle_hr_policy_pdf_2012.pdf
Upload:  MarkdownPageMarkdownResultPage(markdown='Policy\nMandatory\nSeptember 2012\n\nNestlé logo\n\nGood Food, Good Life\n\nPo\n\n# The Nestlé Human Resources Policy\n\nA group of diverse professionals collaborating around a table with documents', page_number=1, success=True, footer=None, header=None)
Total Embddings:  1
Upload:  MarkdownPageMarkdownResultPage(markdown='Policy\nMandatory\nSeptember 2012\n\n**Issuing departement**\n\nHuman Resources\n\n**Target audience**\n\nAll employees\n\n**Approver**\n\nExecutive Board, Nestlé S.A.\n\n**Repository**\n\nAll Nestlé Principles and Policies, Standards and Guidelines can be found in the Centre online repository at: [http://intranet.nestle.com/nestledocs](http://intranet.nestle.com/nestledocs)\n\n**Copyright and confidentiality**\n\nAll rights belong to Nestec Ltd., Vevey, Switzerland. © 2012, Nestec Ltd.

True

In [18]:
# Initiate Chat with Agent

exit_flag = False

# Press 'Escape' Key to End the While Loop below
def on_key_press(event):
    if event.name == 'esc':
        print("Escape key pressed! Exiting input.")
        global exit_flag
        exit_flag = True
        return True  # Stop the keyboard listener

# Keyboard Listener
keyboard.on_press(on_key_press)

# Initialise all LLMs
llama_model = "llama3.1:8b-instruct-q4_K_M"
rag_llm = create_rag_llm(llama_model)

print("Hi, I am HR Policy Assistant. Please ask your question.", flush=True)
while(not exit_flag):
    user_input = input("User: ")

    print("User: ", user_input, flush=True)
    if (not exit_flag):
        user_prompt = {"messages": [HumanMessage(content=user_input)]}
        print("Assistant - ", search_hrpolicy_data(user_input, rag_llm), flush=True)


Hi, I am HR Policy Assistant. Please ask your question.
User:  Get me Information on Leave Policy
Searching Please wait...
Assistant -  I could not find that information. The retrieved contexts do not mention anything about leave policies, but they do discuss employment conditions, total rewards, employee relations, and organizational structure. If you would like to ask a different question or provide more context, I'll be happy to help.
User:  What about Travel Policy ?
Searching Please wait...
Assistant -  I could not find that information in the Retrieved Contexts.
User:  Ok, what about Training Policy ?
Searching Please wait...
Assistant -  Answer: 
The Company determines training and development priorities. The responsibility for turning these into actions is shared between employees, line managers, and the Human Resources.

Experience and on-the-job training are the primary source of learning. Managers are responsible for guiding and coaching employees to succeed in their current